In [ ]:
"""
Project 01: Credit Risk Prediction
Step 05: Evaluation & ROC Curves
"""

In [ ]:
import warnings

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import json
import time
from sklearn.metrics import (
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

In [ ]:
BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if "__file__" in globals() else (os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
PROCESSED = os.path.join(BASE, "data", "processed")
MODELS = os.path.join(BASE, "models")
CHARTS = os.path.join(BASE, "charts")

In [ ]:
os.makedirs(CHARTS, exist_ok=True)

In [ ]:
print("=" * 60)
print("STEP 05: Evaluation & ROC Curves")
print("=" * 60)
start_time = time.time()

In [ ]:
# Load data
df = pd.read_csv(os.path.join(PROCESSED, "preprocessed.csv"))
TARGET = "TARGET"
ID_COL = "SK_ID_CURR"
X = df.drop(columns=[TARGET, ID_COL], errors="ignore")
y = df[TARGET]

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Load models
models = {}
model_files = {
    "Logistic Regression": "logistic_regression.pkl",
    "LightGBM": "lightgbm.pkl",
    "XGBoost": "xgboost.pkl",
}
for name, fname in model_files.items():
    path = os.path.join(MODELS, fname)
    if os.path.exists(path):
        with open(path, "rb") as f:
            data = pickle.load(f)
        if isinstance(data, dict) and "model" in data:
            models[name] = data["model"]
        else:
            models[name] = data
        print(f"Loaded: {name}")

In [ ]:
# Load results
with open(os.path.join(MODELS, "results.json")) as f:
    results = json.load(f)

In [ ]:
# ── ROC Curves ──
print("\n[1/4] Generating ROC curves...")
fig, ax = plt.subplots(figsize=(10, 8))
colors = {"Logistic Regression": "#6C63FF", "LightGBM": "#FF6584", "XGBoost": "#43E97B"}

In [ ]:
for name, model in models.items():
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        auroc = auc(fpr, tpr)
        ax.plot(
            fpr,
            tpr,
            color=colors.get(name, "#888"),
            lw=2,
            label=f"{name} (AUROC={auroc:.4f})",
        )

In [ ]:
ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves — Credit Risk Models", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "roc_curves.png"), dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: roc_curves.png")

In [ ]:
# ── Precision-Recall Curves ──
print("[2/4] Generating PR curves...")
fig, ax = plt.subplots(figsize=(10, 8))

In [ ]:
for name, model in models.items():
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
        precision, recall, _ = precision_recall_curve(y_test, y_proba)
        ap = average_precision_score(y_test, y_proba)
        ax.plot(
            recall,
            precision,
            color=colors.get(name, "#888"),
            lw=2,
            label=f"{name} (AP={ap:.4f})",
        )

In [ ]:
ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title(
    "Precision-Recall Curves — Credit Risk Models", fontsize=14, fontweight="bold"
)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "pr_curves.png"), dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: pr_curves.png")

In [ ]:
# ── Confusion Matrices ──
print("[3/4] Generating confusion matrices...")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

In [ ]:
for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=ax,
        cbar=False,
        xticklabels=["Repaid", "Default"],
        yticklabels=["Repaid", "Default"],
    )
    ax.set_title(f"{name}", fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

In [ ]:
plt.suptitle("Confusion Matrices", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(
    os.path.join(CHARTS, "confusion_matrices.png"), dpi=150, bbox_inches="tight"
)
plt.close()
print("  Saved: confusion_matrices.png")

In [ ]:
# ── Model Comparison Bar Chart ──
print("[4/4] Generating model comparison chart...")
df_res = pd.DataFrame(results).T.reset_index().rename(columns={"index": "Model"})
df_res = df_res.sort_values("auroc", ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
bar_colors = ["#6C63FF", "#FF6584", "#43E97B"]

In [ ]:
for ax, metric, title in zip(
    axes, ["auroc", "f1", "precision", "recall"], ["AUROC", "F1", "Precision", "Recall"]
):
    bars = ax.bar(df_res["Model"], df_res[metric], color=bar_colors, alpha=0.85)
    ax.set_title(title, fontweight="bold")
    ax.set_ylim(0, 1)
    for bar, val in zip(bars, df_res[metric]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{val:.3f}",
            ha="center",
            fontsize=10,
            fontweight="bold",
        )
    ax.tick_params(axis="x", rotation=30)

In [ ]:
plt.suptitle("Model Comparison — All Metrics", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(
    os.path.join(CHARTS, "model_comparison_all.png"), dpi=150, bbox_inches="tight"
)
plt.close()
print("  Saved: model_comparison_all.png")

In [ ]:
elapsed = time.time() - start_time
print(f"\n{'=' * 60}")
print(f"STEP 05 COMPLETE | Time: {elapsed:.1f}s")
print(f"{'=' * 60}")